# Cost optimisation algorithms

**Data sources:** 

* production - solarEdge (ID:1567917)
* consumption - mvm (GroupID: 4)
* price - HUPX.hu



Using cassandra to store the aforementioned sources after transformations: 
* **consumer_profile**
*  **solar_panel_production**
*  **hupx_energy_price**

Data is stored in 15 minute time intervals. production and consumption is stored in **kwh**, price is in **mwh**



In [22]:
spark.version

'3.5.0'

In [1]:
import findspark
import pyspark.sql
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp, col, to_utc_timestamp, count, max, monotonically_increasing_id, lit, concat, expr, regexp_replace,lpad
 

findspark.init()

spark = SparkSession.builder \
    .appName("CassandraConncection") \
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0")\
    .config("spark.sql.catalog.client","com.datastax.spark.connector.datasource.CassandraCatalog") \
    .config("spark.sql.catalog.client.spark.cassandra.connection.host","127.0.0.1")\
    .getOrCreate()
    #.config("spark.cassandra.connection.port", "9042")\
    


24/06/04 09:06:37 WARN Utils: Your hostname, PySpark resolves to a loopback address: 127.0.1.1; using 192.168.64.129 instead (on interface ens33)
24/06/04 09:06:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/student/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/student/.ivy2/cache
The jars for the packages stored in: /home/student/.ivy2/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-efbc916b-7c72-48f9-a75d-cfcb337786bb;1.0
	confs: [default]
	found com.datastax.spark#spark-cassandra-connector_2.12;3.5.0 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.5.0 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found com.datastax.oss#java-driver-core-shaded;4.13.0 in central
	found com.datastax.oss#native-protocol;1.5.0 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central
	found com.typesafe#config;1.4.1 in central
	found org.slf4j#slf4j-api;1.7.26 in central
	found io.dropwizard.metrics#metrics-core;4.1.18 in central
	found org.hdrhistogram#HdrHistogram;2.1.12 in central
	found org.reactivestreams#reactive-streams;1.0.3

# Extract


In [4]:

#reading and transforming csv input, only run this block when needed.
consumption = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",";")\
          .load("../ingest/consumption_profile.csv")

production = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",",")\
          .load("../ingest/production_profile.csv")

price = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",";")\
          .option("inferSchema","true")\
          .load("../ingest/price.csv")

consumption.show()
production.show()
price.show()






+----------+----------+--------+
|     Dátum|Negyedórák| Group#4|
+----------+----------+--------+
|2022.01.01|      0:00|0.018103|
|2022.01.01|      0:15|0.018038|
|2022.01.01|      0:30|0.018010|
|2022.01.01|      0:45|0.018095|
|2022.01.01|      1:00|0.018187|
|2022.01.01|      1:15|0.018158|
|2022.01.01|      1:30|0.018087|
|2022.01.01|      1:45|0.018103|
|2022.01.01|      2:00|0.018172|
|2022.01.01|      2:15|0.018167|
|2022.01.01|      2:30|0.018129|
|2022.01.01|      2:45|0.018056|
|2022.01.01|      3:00|0.017923|
|2022.01.01|      3:15|0.017764|
|2022.01.01|      3:30|0.017725|
|2022.01.01|      3:45|0.017770|
|2022.01.01|      4:00|0.017912|
|2022.01.01|      4:15|0.018011|
|2022.01.01|      4:30|0.018170|
|2022.01.01|      4:45|0.018460|
+----------+----------+--------+
only showing top 20 rows

+----------------+--------------+
|            time|production_(W)|
+----------------+--------------+
|01.01.2022 00:00|             0|
|01.01.2022 00:15|             0|
|01.01.2022 

# Transform

Transformations are needed: All timezones should be casted to **UTC**, granuality should be set to 15 minutes to all sources

production profile:

In [5]:
spark.conf.set("spark.sql.session.timeZone", "UTC") # necessary config to aviod clock shift
production_final = production.withColumn("timestamp",to_timestamp(col("time"),"dd.MM.yyyy HH:mm"))\
                        .withColumn("profile_id", lit(1567917))\
                        .withColumn("production_(W)", col("production_(W)")*0.00025)\
                        .withColumnRenamed("production_(W)", "production_kwh")\
                        .select("profile_id","timestamp","production_kwh")
                        




SyntaxError: unexpected character after line continuation character (3479980726.py, line 4)

consumption profile:

In [16]:
spark.conf.set("spark.sql.legacy.timeParserPolicy","LEGACY")
cons_final = consumption.withColumn("time", to_timestamp(concat(col("Dátum"), lit(" "), col("Negyedórák")), "yyyy.MM.dd HH:mm"))\
                        .withColumn("utc_time", to_utc_timestamp(col("time"), "Europe/Budapest"))\
                        .withColumn("consumption_kwh", col("Group#4").cast("double")*5)\
                        .withColumn("rownum", monotonically_increasing_id())

cons_filter = cons_final.groupBy("utc_time").agg(count("*").alias("count")).filter(col("count")>=2)
cons_filtered = cons_final.join(cons_filter,"utc_time","inner").groupBy("utc_time").agg(max("rownum").alias("rownum"))\
                          .withColumn("modified_utc_time", expr("utc_time + INTERVAL 1 HOUR"))\
                          .select("modified_utc_time","rownum")
cons_final = cons_final.join(cons_filtered,"rownum","left_outer")\
                       .withColumn("timestamp", expr("IFNULL(modified_utc_time, utc_time)"))\
                       .withColumn("profile_id", lit(4))\
                       .select("profile_id","timestamp","consumption_kwh")

cons_final.show()

+----------+-------------------+-------------------+
|profile_id|          timestamp|    consumption_kwh|
+----------+-------------------+-------------------+
|         4|2021-12-31 23:00:00|0.09051500000000001|
|         4|2021-12-31 23:15:00|0.09018999999999999|
|         4|2021-12-31 23:30:00|            0.09005|
|         4|2021-12-31 23:45:00|           0.090475|
|         4|2022-01-01 00:00:00|0.09093499999999999|
|         4|2022-01-01 00:15:00|0.09079000000000001|
|         4|2022-01-01 00:30:00|0.09043499999999999|
|         4|2022-01-01 00:45:00|0.09051500000000001|
|         4|2022-01-01 01:00:00|            0.09086|
|         4|2022-01-01 01:15:00|           0.090835|
|         4|2022-01-01 01:30:00|           0.090645|
|         4|2022-01-01 01:45:00|            0.09028|
|         4|2022-01-01 02:00:00|           0.089615|
|         4|2022-01-01 02:15:00|            0.08882|
|         4|2022-01-01 02:30:00|0.08862500000000001|
|         4|2022-01-01 02:45:00|0.088850000000

price:

In [17]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
price_final = price.withColumn("Hours",regexp_replace("Hours","[HB]",""))\
                   .withColumn("Hours", expr("cast(Hours as int) - 1"))\
                   .withColumn("Quarters",expr("explode(array(':00',':15',':30',':45'))"))\
                   .withColumn("Hours", concat(lpad("Hours",2,"0"),"Quarters"))\
                   .withColumn("time",to_timestamp(concat("Delivery day",lit(" "),"Hours"),"dd.MM.yyyy HH:mm"))\
                   .withColumn("utc_timestamp", to_utc_timestamp("time", "Europe/Budapest"))\
                   .withColumn("rownum", monotonically_increasing_id())

price_filter = price_final.groupBy("utc_timestamp").agg(count("*").alias("count")).filter(col("count")>=2)
price_filtered = price_final.join(price_filter,"utc_timestamp","inner").groupBy("utc_timestamp").agg(max("rownum").alias("rownum"))\
                            .withColumn("modified_utc_timestamp", expr("utc_timestamp + INTERVAL 1 HOUR")).select("modified_utc_timestamp","rownum")
price_final = price_final.join(price_filtered,"rownum","left_outer")\
                         .withColumn("timestamp", expr("IFNULL(modified_utc_timestamp, utc_timestamp)"))\
                         .withColumnRenamed("Prices (EUR/Mwh)","price_eur_per_mwh")\
                         .select("timestamp","price_eur_per_mwh")

                   

price_final.show()
price_final.printSchema()
price_final.count()

+-------------------+-----------------+
|          timestamp|price_eur_per_mwh|
+-------------------+-----------------+
|2021-12-31 23:00:00|            61.84|
|2021-12-31 23:15:00|            61.84|
|2021-12-31 23:30:00|            61.84|
|2021-12-31 23:45:00|            61.84|
|2022-01-01 00:00:00|            41.33|
|2022-01-01 00:15:00|            41.33|
|2022-01-01 00:30:00|            41.33|
|2022-01-01 00:45:00|            41.33|
|2022-01-01 01:00:00|            43.22|
|2022-01-01 01:15:00|            43.22|
|2022-01-01 01:30:00|            43.22|
|2022-01-01 01:45:00|            43.22|
|2022-01-01 02:00:00|            45.46|
|2022-01-01 02:15:00|            45.46|
|2022-01-01 02:30:00|            45.46|
|2022-01-01 02:45:00|            45.46|
|2022-01-01 03:00:00|            37.67|
|2022-01-01 03:15:00|            37.67|
|2022-01-01 03:30:00|            37.67|
|2022-01-01 03:45:00|            37.67|
+-------------------+-----------------+
only showing top 20 rows

root
 |-- time

70080

# Load

In [30]:
cons_final.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="consumption_profile", keyspace="energycom") \
    .save()

In [9]:
production_final.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="production_profile", keyspace="energycom") \
    .save()

In [10]:
price_final.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="hupx_energy_price", keyspace="energycom") \
    .save()

# Reading from Cassandra DB


In [24]:
consumer = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="consumption_profile", keyspace="energycom") \
    .load()
producer = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="production_profile", keyspace="energycom") \
    .load()
price = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="hupx_energy_price", keyspace="energycom") \
    .load()

consumer.show()
producer.show()
price.show()

+-------------------+----------+---------------+
|          timestamp|profile_id|consumption_kwh|
+-------------------+----------+---------------+
|2022-08-21 23:45:00|         4|        0.01166|
|2023-05-31 11:30:00|         4|       0.040605|
|2022-01-12 00:15:00|         4|       0.018534|
|2023-11-20 10:30:00|         4|       0.072211|
|2022-02-14 19:00:00|         4|       0.018955|
|2023-08-11 21:45:00|         4|       0.011796|
|2022-04-17 16:30:00|         4|       0.014745|
|2022-09-13 23:00:00|         4|       0.012007|
|2022-03-22 18:15:00|         4|       0.019435|
|2022-04-06 10:00:00|         4|        0.04597|
|2022-09-09 09:15:00|         4|       0.052922|
|2022-04-25 17:15:00|         4|        0.01548|
|2022-03-08 21:15:00|         4|       0.015874|
|2022-12-26 17:15:00|         4|       0.017097|
|2022-04-30 03:30:00|         4|       0.015528|
|2023-04-11 21:30:00|         4|        0.01444|
|2023-12-22 11:30:00|         4|       0.062325|
|2023-12-25 11:45:00

# Combining data

In [19]:
from pyspark.sql.functions import *

# filter
consumer   = consumer.filter(col("profile_id") == 4)
production = production.filter(col("profile_id") == 1567917)

# join
process = consumer.select("timestamp","consumption_kwh").join(production.select("timestamp","production_kwh"),"timestamp","inner")\
                  .join(price.select("timestamp","price_eur_per_mwh"),"timestamp","inner")

#sort and cast to Pandas
process = process.orderBy("timestamp")\
                 .withColumn("consumption_kwh", process["consumption_kwh"].cast("float"))\
                 .withColumn("production_kwh", process["production_kwh"].cast("float"))\
                 .withColumn("price_eur_per_mwh", process["price_eur_per_mwh"].cast("float"))\
                 .toPandas()
process.head()


,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh
0,2022-01-01 00:00:00,0.090935,0.0,41.330002
1,2022-01-01 00:15:00,0.090790,0.0,41.330002
2,2022-01-01 00:30:00,0.090435,0.0,41.330002
3,2022-01-01 00:45:00,0.090515,0.0,41.330002
4,2022-01-01 01:00:00,0.090860,0.0,43.220001


# Optimistaion strategies

Goal: minimise **energy costs**

Scenarios:
* only pv
* greedy strategy
* rule-based strategy full
* rule-based strategy without LT

# PV only

In [78]:
pv = process.assign(feeding_grid = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, (x['production_kwh'] - x['consumption_kwh'])*0.25, 0),
                    taking_from_grid   = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, (x['consumption_kwh'] - x['production_kwh'])*0.25, 0),
                    sell_price = lambda x: x["feeding_grid"]*x["price_eur_per_mwh"]*0.00025,
                    buy_price = lambda x: x["taking_from_grid"]*x["price_eur_per_mwh"]*0.00025,
                    net_revenue = lambda x: x["sell_price"] - x["buy_price"])

pv.head()

,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh,feeding_grid,taking_from_grid,sell_price,buy_price,net_revenue
0,2022-01-01 00:00:00,0.018187,0.0,41.330002,0.0,0.004547,0.0,0.000047,-0.000047
1,2022-01-01 00:15:00,0.018158,0.0,41.330002,0.0,0.004539,0.0,0.000047,-0.000047
2,2022-01-01 00:30:00,0.018087,0.0,41.330002,0.0,0.004522,0.0,0.000047,-0.000047
3,2022-01-01 00:45:00,0.018103,0.0,41.330002,0.0,0.004526,0.0,0.000047,-0.000047
4,2022-01-01 01:00:00,0.018172,0.0,43.220001,0.0,0.004543,0.0,0.000049,-0.000049


# Greedy strategy
If production covers consumption, then surplus is stored into BESS, if it is full, then the remainder is sold to the grid.
If 

Másik esetben, mikor nem fedezi a termelés a fogyasztást, akkor a szükséges energiamennyiséget első sorban az akkumulátorban tárolt energiából pótoljuk, hogyha az sem elég, akkor vételezünk a hálózatról.


In [12]:
Capacity = 100
Charge_rate = 50
Discharge_rate = 50
Capacity_min = 0


In [75]:
import numpy as np
# energia többlet és igény kiszámítása a termelés és fogyasztás különbségéből,
# valamint töltöttséget követő oszlopok bevezetése a negyedóra elejére és végére, alapértelmezett értékadás.
basic_p = process.assign(energy_to_store = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, (x['production_kwh'] - x['consumption_kwh'])*0.25, 0),
                       energy_to_get   = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, (x['consumption_kwh'] - x['production_kwh'])*0.25, 0),
                       battery_charge_start = float(0.00000),
                       battery_charge_end   = float(0.00000))
basic_p.head()

#from pyspark.sql.functions import when
#basic_p = process.withColumn("energy_to_store", when(col("production_kwh")-col("consumption_kwh")>0,col("production_kwh")-col("consumption_kwh"))\
#                          .otherwise(0))\
#               .withColumn("energy_to_get",when(col("consumption_kwh")-col("production_kwh")>0,col("consumption_kwh")-col("production_kwh"))\
#                          .otherwise(0))\
#               .withColumn("battery_charge_at_start",lit(0.00000))\
#               .withColumn("battery_charge_at_end",lit(0.00000))

# kezdő értéket állíthatunk az akkumulátorunknak a vizsgált időszak elején.
# basic_p.at[0,"battery_charge_start"] = 2

#ha tárolni kell akkor mennyit tudunk eltárolni
if basic_p.at[0,'energy_to_store'] > 0:
        temp = basic_p.at[0,'battery_charge_start'] + basic_p.at[0,'energy_to_store']
        if temp <= capacity:
            basic_p.at[0,'battery_charge_end'] = temp
        else:
            basic_p.at[0,'battery_charge_end'] = capacity
            
#ha be kell szerezni, mennyit tudunk szolgáltatni saját tárunkból
else:
        temp = basic_p.at[0,'battery_charge_start'] - basic_p.at[0,'energy_to_get']
        if temp > 0:
            basic_p.at[0,'battery_charge_end'] = temp
        else:
            basic_p.at[0,'battery_charge_end'] = 0

for i in range(1,int(basic_p.size/8)):
    basic_p.at[i,'battery_charge_start'] = basic_p.at[i-1,'battery_charge_end']
    if basic_p.at[i,'energy_to_store'] > 0:
        temp = basic_p.at[i,'battery_charge_start'] + basic_p.at[i,'energy_to_store']
        if temp <= capacity:
            basic_p.at[i,'battery_charge_end'] = temp
        else:
            basic_p.at[i,'battery_charge_end'] = capacity
    else:
        temp = basic_p.at[i,'battery_charge_start'] - basic_p.at[i,'energy_to_get']
        if temp > 0:
            basic_p.at[i,'battery_charge_end'] = temp
        else:
            basic_p.at[i,'battery_charge_end'] = 0
#basic_p.tail(10)

#az előző adatok alapján megállapítani, hogy mennyit fogyasztunk saját, illetve hálózati energiából, 
# valamint mennyi energiát táplálunk be saját akkumulátorunkba, vagy a hálózatba. 
basic_p = basic_p.assign(taking_from_battery = lambda x: np.where(x['battery_charge_start'] - x['battery_charge_end'] > 0, x['battery_charge_start'] - x['battery_charge_end'], 0),
                       feeding_battery   = lambda x: np.where(x['battery_charge_end'] - x['battery_charge_start'] > 0, x['battery_charge_end'] - x['battery_charge_start'], 0),
                       taking_from_grid  = lambda x: x['energy_to_get']-x['taking_from_battery'],
                       feeding_grid      = lambda x: x['energy_to_store']-x['feeding_battery'],
                       price             = lambda x: (x['taking_from_grid']*x['price_eur_per_mwh']-x['feeding_grid']*x['price_eur_per_mwh'])*0.00025) 

basic_p.head(100)



,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh,energy_to_store,energy_to_get,battery_charge_start,battery_charge_end,taking_from_battery,feeding_battery,taking_from_grid,feeding_grid,price
0,2022-01-01 00:00:00,0.018187,0.0,41.330002,0.0,0.004547,0.000000,0.000000,0.000000,0.0,0.004547,0.0,0.000047
1,2022-01-01 00:15:00,0.018158,0.0,41.330002,0.0,0.004539,0.000000,0.000000,0.000000,0.0,0.004539,0.0,0.000047
2,2022-01-01 00:30:00,0.018087,0.0,41.330002,0.0,0.004522,0.000000,0.000000,0.000000,0.0,0.004522,0.0,0.000047
3,2022-01-01 00:45:00,0.018103,0.0,41.330002,0.0,0.004526,0.000000,0.000000,0.000000,0.0,0.004526,0.0,0.000047
4,2022-01-01 01:00:00,0.018172,0.0,43.220001,0.0,0.004543,0.000000,0.000000,0.000000,0.0,0.004543,0.0,0.000049
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2022-01-01 23:45:00,0.018095,0.0,57.080002,0.0,0.004524,2.314730,2.310206,0.004524,0.0,0.000000,0.0,0.000000
96,2022-01-02 00:00:00,0.018187,0.0,52.590000,0.0,0.004547,2.310206,2.305659,0.004547,0.0,0.000000,0.0,0.000000
97,2022-01-02 00:15:00,0.018158,0.0,52.590000,0.0,0.004539,2.305659,2.301120,0.004539,0.0,0.000000,0.0,0.000000
98,2022-01-02 00:30:00,0.018087,0.0,52.590000,0.0,0.004522,2.301120,2.296598,0.004522,0.0,0.000000,0.0,0.000000


In [77]:
simple_total_generation = basic_p.assign(generation = lambda x: x['production_kwh']*0.25)['generation'].sum()
simple_total_self_consumed = basic_p.assign(self_consumed = lambda x: x['production_kwh']*0.25 -x['feeding_grid'])['self_consumed'].sum()
print("Total generation: ",simple_total_generation, " kw")
print("Total_self_consumed: ", simple_total_self_consumed, " kw")
print(basic_p['consumption_kwh'].sum())

Total generation:  7102.7144  kw
Total_self_consumed:  509.71448887999213  kw
1999.9276


# Rule Based Strategy


In [20]:
rule_based = process.assign(Surplus = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, x['production_kwh'] - x['consumption_kwh'], 0),                       Deficit  = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, x['consumption_kwh'] - x['production_kwh'], 0),
                       SOC_start     = float(0.00000),
                       SOC_end       = float(0.00000),
                       P_P2L         = float(0.00000),
                       P_P2B         = float(0.00000),
                       P_P2G         = float(0.00000),
                       P_G2B         = float(0.00000),
                       P_G2L         = float(0.00000),
                       P_B2L         = float(0.00000),
                       Buy_price     = float(0.00000),
                       Selling_price = float(0.0000))
n_ut = 1
n_lt = 1

#first row
UT = rule_based.iloc[0 : 96]['price_eur_per_mwh'].quantile(0.75, interpolation='higher')
LT = 0

if rule_based.at[0,"Deficit"] > 0:
        #P_P2B = 0
        #P_P2G = 0
        rule_based.at[0,"P_P2L"] = rule_based.at[0,"production_kwh"]
        
        # cheaper than UT
        if rule_based.at[0,"price_eur_per_mwh"] < UT:
            #P_B2L = 0
            rule_based.at[0,"P_G2L"] = rule_based.at[0,"Deficit"]

            #cheaper than LT and there is space
            if (rule_based.at[0,"price_eur_per_mwh"] < LT) & (rule_based.at[0,"SOC_start"] < Capacity):
               rule_based.at[0,"P_G2B"] = __builtins__.min([Capacity - rule_based.at[0,"SOC_start"],Charge_rate*0.25])
                #else P_G2B = 0
        # expensive
        else:
            # P_G2B = 0

            # BESS not empty
            if rule_based.at[0,"SOC_start"] > Capacity_min:

                #if rule_based.at[i,"Deficit"] < Discharge_rate:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min])
                #
                #else:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,Discharge_rate, rule_based.at[i,"SOC_start"]- Capacity_min])
                rule_based.at[0,"P_B2L"] = __builtins__.min([rule_based.at[0,"Deficit"], rule_based.at[0,"SOC_start"]- Capacity_min, Discharge_rate*0.25])
                rule_based.at[0,"P_G2L"] = rule_based.at[0,"Deficit"] - rule_based.at[0,"P_B2L"]
            #BESS is empty
            else:
                #P_B2L = 0
                rule_based.at[0,"P_G2L"] = rule_based.at[0,"Deficit"]
                
    # Surplus side            
else:
        #P_G2L = 0
        #P_B2L = 0
        rule_based.at[0,"P_P2L"] = rule_based.at[0,"consumption_kwh"]

        # is BESS full?
        if rule_based.at[0,"SOC_start"] < Capacity:

            # Is surplus < Charge rate
            if rule_based.at[0,"Surplus"] < Charge_rate:
                
                rule_based.at[0,"P_P2B"] = __builtins__.min(rule_based.at[0,"Surplus"], Capacity - rule_based.at[0,"SOC_start"])
                rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"] - rule_based.at[0,"P_P2B"]
                
                if rule_based.at[0,"price_eur_per_mwh"] < LT:
                    rule_based.at[0,"P_G2B"] = __builtins__.min(Charge_rate - rule_based.at[0,"P_P2B"], Capacity - rule_based.at[0,"SOC_start"] - rule_based.at[0,"P_P2B"])
                    # else P_G2B = 0
            
            # surplus bigger
            else:
                # P_G2B = 0
                rule_based.at[0,"P_P2B"] = __builtins__.min(Charge_rate, Capacity - rule_based.at[0,"SOC_start"])
                rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"] - rule_based.at[0,"P_P2B"]
                
        # BESS is full
        else:
            #P_P2B = 0
            #P_G2B = 0
            rule_based.at[0,"P_P2G"] = rule_based.at[0,"Surplus"]

#P_P2L

#P_P2G
#P_G2B
#P_G2L


rule_based.at[0,"SOC_end"] = rule_based.at[0,"SOC_start"] + rule_based.at[0,"P_P2B"] + rule_based.at[0,"P_G2B"] - rule_based.at[0,"P_B2L"]
rule_based.at[0,"Buy_price"] = (rule_based.at[0,"P_G2B"] + rule_based.at[0,"P_G2L"])*rule_based.at[0,"price_eur_per_mwh"]*0.00025
rule_based.at[0,"Selling_price"] = rule_based.at[0,"P_P2G"]*rule_based.at[0,"price_eur_per_mwh"]*0.00025

for i in range(1,int(rule_based.size/16)):
    # setting SOC 
    rule_based.at[i,"SOC_start"] = rule_based.at[i-1,"SOC_end"]
    # calculating thresholds
    if (n_ut % 96 == 0) & (n_ut != rule_based.size/16):
        UT = rule_based.iloc[i : i + 96]['price_eur_per_mwh'].quantile(0.75, interpolation='higher')
    if (n_lt % 96) & (n_lt >= 1440):
        LT = rule_based.iloc[i - 1439 : i]['price_eur_per_mwh'].quantile(0.25, interpolation='lower')
    n_ut += 1
    n_lt += 1

    # comparing demand and production
    # Deficit side
    if rule_based.at[i,"Deficit"] > 0:
        #P_P2B = 0
        #P_P2G = 0
        rule_based.at[i,"P_P2L"] = rule_based.at[i,"production_kwh"]
        
        # cheaper than UT
        if rule_based.at[i,"price_eur_per_mwh"] < UT:
            #P_B2L = 0
            rule_based.at[i,"P_G2L"] = rule_based.at[i,"Deficit"]

            #cheaper than LT and there is space
            if (rule_based.at[i,"price_eur_per_mwh"] < LT) & (rule_based.at[i,"SOC_start"] < Capacity):
               rule_based.at[i,"P_G2B"] = __builtins__.min([Capacity - rule_based.at[i,"SOC_start"],Charge_rate*0.25])
                #else P_G2B = 0
        # expensive
        else:
            # P_G2B = 0

            # BESS not empty
            if rule_based.at[i,"SOC_start"] > Capacity_min:

                #if rule_based.at[i,"Deficit"] < Discharge_rate:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min])
                #
                #else:
                #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,Discharge_rate, rule_based.at[i,"SOC_start"]- Capacity_min])
                rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"Deficit"], rule_based.at[i,"SOC_start"]- Capacity_min, Discharge_rate*0.25])
                rule_based.at[i,"P_G2L"] = rule_based.at[i,"Deficit"] - rule_based.at[i,"P_B2L"]
            #BESS is empty
            else:
                #P_B2L = 0
                rule_based.at[i,"P_G2L"] = rule_based.at[i,"Deficit"]
                
    # Surplus side            
    else:
        #P_G2L = 0
        #P_B2L = 0
        rule_based.at[i,"P_P2L"] = rule_based.at[i,"consumption_kwh"]

        # is BESS full?
        if rule_based.at[i,"SOC_start"] < Capacity:

            # Is surplus < Charge rate
            if rule_based.at[i,"Surplus"] < Charge_rate:
                
                rule_based.at[i,"P_P2B"] = __builtins__.min(rule_based.at[i,"Surplus"], Capacity - rule_based.at[i,"SOC_start"])
                rule_based.at[i,"P_P2G"] = rule_based.at[i,"Surplus"] - rule_based.at[i,"P_P2B"]
                
                if rule_based.at[i,"price_eur_per_mwh"] < LT:
                    rule_based.at[i,"P_G2B"] = __builtins__.min(Charge_rate - rule_based.at[i,"P_P2B"], Capacity - rule_based.at[i,"SOC_start"] - rule_based.at[i,"P_P2B"])
                    # else P_G2B = 0
            
            # surplus bigger
            else:
                # P_G2B = 0
                rule_based.at[i,"P_P2B"] = __builtins__.min(Charge_rate, Capacity - rule_based.at[i,"SOC_start"])
                rule_based.at[i,"P_P2G"] = rule_based.at[i,"Surplus"] - rule_based.at[i,"P_P2B"]
                
        # BESS is full
        else:
            #P_P2B = 0
            #P_G2B = 0
            rule_based.at[i,"P_P2G"] = rule_based.at[i,"Surplus"]

    #executing the calculations
    rule_based.at[i,"SOC_end"]       = rule_based.at[i,"SOC_start"] + rule_based.at[i,"P_P2B"] + rule_based.at[i,"P_G2B"] - rule_based.at[i,"P_B2L"]
    rule_based.at[i,"Buy_price"]     = (rule_based.at[i,"P_G2B"] + rule_based.at[i,"P_G2L"])*rule_based.at[i,"price_eur_per_mwh"]*0.00025
    rule_based.at[i,"Selling_price"] = rule_based.at[i,"P_P2G"]*rule_based.at[i,"price_eur_per_mwh"]*0.00025
            
UT=0
LT=0

print("Buy:")
print(rule_based["Buy_price"].sum())
print("Sell:")
print(rule_based["Selling_price"].sum())
print("Diff:")
print(rule_based["Buy_price"].sum() - rule_based["Selling_price"].sum())
print("Surplus:")
print(rule_based["Surplus"].sum())
print("Deficit:")
print(rule_based["Deficit"].sum())

Buy:
169.2605932246018
Sell:
1049.0800006827228
Diff:
-879.8194074581211
Surplus:
23526.621
Deficit:
5115.3994
